In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(42)

In [ ]:
df = pd.read_csv("/Users/anandsharma/Documents/PhD/LearnML/CampusX/Datasets/fashionmnist/versions/4/fashion-mnist_train.csv")
df.head()

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 images from the dataset", fontsize=16)

for i, ax in enumerate(axes.flatten()):
    image = df.iloc[i, 1:].values.reshape(28, 28)
    label = df.iloc[i, 0]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scaling the data
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# Create CustomDataset class

class CustomDataset(Dataset):

    def __init__(self, features, labels): 
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


In [ ]:
# Create train and test datasets objects
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
# Cretate train and test dataloaders
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True) #, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False) #, pin_memory=True)

In [ ]:
len(train_loader), len(test_loader)

In [ ]:
class MyNN(nn.Module): 

    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 10)        )

    def forward(self, x):
        return self.model(x)   
    

In [ ]:
learning_rate = 0.1
epochs = 100 

In [ ]:
model = MyNN(num_features=X_train.shape[1])
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
for epoch in range(epochs):

    model.train()
    total_epoch_loss = 0

    for batch_idx, batch in enumerate(train_loader):

        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Debug (optional)
        # if batch_idx % 10 == 0:
        #     print(f"MPS allocated: {torch.mps.current_allocated_memory()/1e6:.2f} MB")
        #     print(f"MPS driver: {torch.mps.driver_allocated_memory()/1e6:.2f} MB")

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_epoch_loss/len(train_loader):.4f}")

In [ ]:
model.eval()  # Set the model to evaluation mode

In [ ]:
total = 0
correct = 0

with torch.no_grad():
    for batch in test_loader:
        batch_features, batch_labels = batch
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    print(f"Accuracy: {100 * correct / total:.4f}%")

In [ ]:
total = 0
correct = 0

with torch.no_grad():
    for batch in train_loader:
        batch_features, batch_labels = batch
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    print(f"Accuracy: {100 * correct / total:.4f}%")